In [1]:
import pandas as pd
import numpy as np
import gc
import io
import os
import csv
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils
import modules.encode as encode

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

import lightgbm as lgb

HEAD = 180000
SEED = 71

f001 = [
    'f001_NAME_CONTRACT_TYPE',
    'f001_CODE_GENDER',
    'f001_FLAG_OWN_CAR',
    'f001_FLAG_OWN_REALTY',
    'f001_NAME_TYPE_SUITE',
    'f001_NAME_INCOME_TYPE',
    'f001_NAME_EDUCATION_TYPE',
    'f001_NAME_FAMILY_STATUS',
    'f001_NAME_HOUSING_TYPE',
    'f001_OCCUPATION_TYPE',
    'f001_WEEKDAY_APPR_PROCESS_START',
    'f001_ORGANIZATION_TYPE',
    'f001_FONDKAPREMONT_MODE',
    'f001_HOUSETYPE_MODE',
    'f001_WALLSMATERIAL_MODE',
    'f001_EMERGENCYSTATE_MODE',
]

f002 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]

f003 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]


ALL_CAT = f001 + f002 + f003

In [7]:
used_2 = "4_used_2_f0_f102"
low_corr_target = "2_low_corr_target_f0_f102"

In [4]:
used_2 = "4_used_2_f0_f103"
low_corr_target = "2_low_corr_target_f0_f103"

In [2]:
used_2 = "4_used_2_f0_f104"
low_corr_target = "2_low_corr_target_f0_f104"

In [6]:
used_2 = "4_used_2_f0_f105_f106_f107_f108"
low_corr_target = "2_low_corr_target_f0_f105_f106_f107_f108"

In [7]:
import re
def read_feather_with_head(file_path):
    return pd.read_feather(file_path).head(HEAD)

def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + line.strip() + ".f" for line in f]
        return features
    
def sanitize_feature_name(name):
    return re.sub(r"[+(),. ]", "_", name)

In [ ]:
# param tunning:
param = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6, # or 10 - 15
    'num_leaves': 31, # 63 or 20 - 50
    'max_bin': 255,
    'min_child_weight': 10, # 5 or 10
    'min_data_in_leaf': 150, # 100 or 150 
    'reg_lambda': 1, #0.5 or 0.01 or 0.1 # L2 regularization term on weights.
    'reg_alpha': 0.5, # 0.5  # L1 regularization term on weights.
    'colsample_bytree': 0.7,
    'subsample': 0.6, # 0.5
    # 'nthread': 12,
    'bagging_freq': 1,
    'verbose': 0,
    'seed': SEED,
    # thêm cấu hình cho GPU
    'device_type': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

In [9]:
n_thread=12 # cpu 6 cores 12 threads

# đối với feature có thể dùng được sau khi lọc var vả corr trước
feature_paths = read(ROOT + f"/.log/_used/{used_2}.txt")
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/5_lightgbm_{used_2}.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split'])
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns
        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i]])
        
        writer.writerows(csv_rows)
        print(len(csv_rows))
        chunk+=1

Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.777418	val's auc: 0.749233
[200]	train's auc: 0.7972	val's auc: 0.75613
[300]	train's auc: 0.813101	val's auc: 0.761156
[400]	train's auc: 0.826304	val's auc: 0.763497
[500]	train's auc: 0.837945	val's auc: 0.764985
[600]	train's auc: 0.848519	val's auc: 0.766104
[700]	train's auc: 0.858744	val's auc: 0.766907
[800]	train's auc: 0.868246	val's auc: 0.767264
[900]	train's auc: 0.877282	val's auc: 0.76754
[1000]	train's auc: 0.885834	val's auc: 0.767849
Did not meet early stopping. Best iteration is:
[996]	train's auc: 0.885506	val's auc: 0.767878
600
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.703046	val's auc: 0.652036
[200]	train's auc: 0.72268	val's auc: 0.658707
[300]	train's auc: 0.740216	val's auc: 0.66247
[400]	train's auc: 0.75514	val's auc: 0.664351
[500]	train's auc: 0.769226	val's auc: 0.665621
[600]	train's auc: 0.782171	val's auc: 0.666002
[700]	train's 

### Mô hình phi tuyến nên phải xét cả những feature có tương quan thấp với target

In [10]:
n_thread=12 # cpu 6 cores 12 threads

# đối với feature có thể dùng được sau khi lọc var vả corr trước
feature_paths = read(ROOT + f"/.log/low_corr_target/{low_corr_target}.txt")
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

feature_paths_size = len(feature_paths)

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/6_lightgbm_{low_corr_target}.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split'])
    n = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns

        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i]])
        
        writer.writerows(csv_rows)
        chunk+=1
        
        n += len(file_paths)
        print(n, " / ", feature_paths_size)

Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.728181	val's auc: 0.678662
[200]	train's auc: 0.754358	val's auc: 0.689202
[300]	train's auc: 0.775112	val's auc: 0.695503
[400]	train's auc: 0.792258	val's auc: 0.697999
[500]	train's auc: 0.807754	val's auc: 0.700165
[600]	train's auc: 0.821764	val's auc: 0.70167
[700]	train's auc: 0.834727	val's auc: 0.702785
[800]	train's auc: 0.846425	val's auc: 0.703612
[900]	train's auc: 0.857381	val's auc: 0.70419
[1000]	train's auc: 0.867969	val's auc: 0.704653
Did not meet early stopping. Best iteration is:
[995]	train's auc: 0.867449	val's auc: 0.704658
600  /  1011
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.701152	val's auc: 0.637104
[200]	train's auc: 0.726487	val's auc: 0.644431
[300]	train's auc: 0.748664	val's auc: 0.64943
[400]	train's auc: 0.76767	val's auc: 0.652253
[500]	train's auc: 0.784118	val's auc: 0.654177
[600]	train's auc: 0.798953	val's auc: 0.655556
[7